## Part 1, NLTK

In [17]:
# Importing necessary libraries

import nltk
from nltk.corpus import cmudict
cmu = cmudict.dict()
import pandas as pd
import spacy
import os
from pathlib import Path
import math
from collections import Counter, defaultdict
import re

In [3]:
# Taken from coursework

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2000000

In [4]:
# Function that takes the .txt book files from local 'texts' directory and reads them into a Dataframe

def read_novels(text_dir="texts"):
    rows = []

    for filename in os.listdir(text_dir):
        if not filename.endswith(".txt"):                           # Ignores files that aren't .txt
            continue

        title, author, year = filename[:-len(".txt")].split("-")    # splits the filename off from the file type

        with open(os.path.join(text_dir, filename), "r", encoding="utf-8") as f:
            text = f.read()

        rows.append({
            "text": text,
            "title": title.replace("_", " "),                       # Some formatting for the titles
            "author": author,
            "year": int(year),
        })

    books = pd.DataFrame(rows, columns=["text", "title", "author", "year"])          # Create dataframe and titles
    books = books.sort_values(by="year", ignore_index=True)                          # Sort by year and reset index

    return books

In [5]:
# Sense checking dataframe that was created

books = read_novels() 
books

,text,title,author,year
0,\nCHAPTER 1\n\nThe family of Dashwood had long...,Sense and Sensibility,Austen,1811
1,'Wooed and married and a'.'\n'Edith!' said Mar...,North and South,Gaskell,1855
2,Book the First--Recalled to Life\n\n\n\n\nI. T...,A Tale of Two Cities,Dickens,1858
3,"SAMUEL BUTLER.\nAugust 7, 1901\n\nCHAPTER I: W...",Erewhon,Butler,1872
4,THE AMERICAN\n\nby Henry James\n\n\n1877\n\n\n...,The American,James,1877
5,\nThe Picture of Dorian Gray\n\nby\n\nOscar Wi...,Dorian Gray,Wilde,1890
6,Phase the First: The Maiden\n\n\nI\n\n\nOn an ...,Tess of the DUrbervilles,Hardy,1891
7,THE SECRET GARDEN\n\nBY FRANCES HODGSON BURNET...,The Secret Garden,Burnett,1911
8,Chapter 1\n\nOnce upon a time and a very good ...,Portrait of the Artist,Joyce,1916
9,\nTHE BLACK MOTH\n\nA ROMANCE OF THE XVIII CEN...,The Black Moth,Heyer,1926


In [6]:
# Type token ratio function, returns a dictionary

def nltk_ttr(books):
    ttrs = {}

    for _, row in books.iterrows():
        tokens = nltk.word_tokenize(row["text"])
        words = [token.lower() for token in tokens if token.isalpha()]  # Project asks for no punctuation. This is the simplest way, even if a few contractions are lost
        ttrs[row["title"]] = len(set(words)) / len(words)               # Actual ratio calculation

    return ttrs

In [7]:
# Sense checking the created dictionary

ttrs = nltk_ttr(books)
ttrs

{'Sense and Sensibility': 0.052847302442989776,
 'North and South': 0.0549040694681204,
 'A Tale of Two Cities': 0.07072694469399422,
 'Erewhon': 0.09151270564132943,
 'The American': 0.06381607058523676,
 'Dorian Gray': 0.08355234620193412,
 'Tess of the DUrbervilles': 0.07778957979554696,
 'The Secret Garden': 0.05847231570812455,
 'Portrait of the Artist': 0.10472745625841184,
 'The Black Moth': 0.07866588875923765,
 'Orlando': 0.1137245917497168,
 'Blood Meridian': 0.08568897067593587}

In [8]:
# Function to count syllables

def count_syllables(word):
    word = word.lower()

    if word in cmu:
        return len([phoneme for phoneme in cmu[word][0] if phoneme[-1].isdigit()])

    return len(re.findall(r"[aeiouy]+", word))

In [12]:
# Calculating the Flesch Kinkaid score

def flesch_kincaid(books):
    fk_scores = {}

    for _, row in books.iterrows():
        text = row["text"]

        sentences = nltk.sent_tokenize(text)                                                   # Tokenizing sentences
        words = [token.lower() for token in nltk.word_tokenize(text) if token.isalpha()]       # Tokenizing words

        num_sentences = len(sentences)
        num_words = len(words)
        num_syllables = sum(count_syllables(word) for word in words)

        score = 206.835 - 1.015 * (num_words / num_sentences) - 84.6 * (num_syllables / num_words)    # FK reading level score formula
        fk_scores[row["title"]] = score

    return fk_scores

In [13]:
flesch_kincaid(books)

{'Sense and Sensibility': 60.05908130639207,
 'North and South': 76.66030517141499,
 'A Tale of Two Cities': 67.19889966376635,
 'Erewhon': 51.84305241708813,
 'The American': 71.60520189572668,
 'Dorian Gray': 81.02336183614686,
 'Tess of the DUrbervilles': 72.83723176816379,
 'The Secret Garden': 85.41417951846584,
 'Portrait of the Artist': 76.43945662819628,
 'The Black Moth': 83.43586574508568,
 'Orlando': 67.55542985791602,
 'Blood Meridian': 81.85477404065995}

In [ ]:
def parse(df, store_path="texts/parsed.pickle"):
    df["parsed"] = df["text"].apply(nlp)
    df.to_pickle(store_path)

    return df

In [ ]:
parse(books)

In [16]:
books_spacy = pd.read_pickle("texts/parsed.pickle")
books_spacy

,text,title,author,year,parsed
0,\nCHAPTER 1\n\nThe family of Dashwood had long...,Sense and Sensibility,Austen,1811,"(\n, CHAPTER, 1, \n\n, The, family, of, Dashwo..."
1,'Wooed and married and a'.'\n'Edith!' said Mar...,North and South,Gaskell,1855,"(', Wooed, and, married, and, a, ', ., ', \n, ..."
2,Book the First--Recalled to Life\n\n\n\n\nI. T...,A Tale of Two Cities,Dickens,1858,"(Book, the, First, --, Recalled, to, Life, \n\..."
3,"SAMUEL BUTLER.\nAugust 7, 1901\n\nCHAPTER I: W...",Erewhon,Butler,1872,"(SAMUEL, BUTLER, ., \n, August, 7, ,, 1901, \n..."
4,THE AMERICAN\n\nby Henry James\n\n\n1877\n\n\n...,The American,James,1877,"(THE, AMERICAN, \n\n, by, Henry, James, \n\n\n..."
5,\nThe Picture of Dorian Gray\n\nby\n\nOscar Wi...,Dorian Gray,Wilde,1890,"(\n, The, Picture, of, Dorian, Gray, \n\n, by,..."
6,Phase the First: The Maiden\n\n\nI\n\n\nOn an ...,Tess of the DUrbervilles,Hardy,1891,"(Phase, the, First, :, The, Maiden, \n\n\n, I,..."
7,THE SECRET GARDEN\n\nBY FRANCES HODGSON BURNET...,The Secret Garden,Burnett,1911,"(THE, SECRET, GARDEN, \n\n, BY, FRANCES, HODGS..."
8,Chapter 1\n\nOnce upon a time and a very good ...,Portrait of the Artist,Joyce,1916,"(Chapter, 1, \n\n, Once, upon, a, time, and, a..."
9,\nTHE BLACK MOTH\n\nA ROMANCE OF THE XVIII CEN...,The Black Moth,Heyer,1926,"(\n, THE, BLACK, MOTH, \n\n, A, ROMANCE, OF, T..."


In [18]:
def syntactic_subs(text, n = 10):
    '''Function that counts the 10 most common syntactic subjects for a text'''

    subjects = [
        token.text.lower()
        for token in text
        if token.dep_ in ('nsubj', 'nsubjpass')
    ]
    return Counter(subjects).most_common(n)

In [ ]:
def pmi_calc(text, subject, n = 10):
    '''
    This function takes a subject and calculates  most associated verbs by pointwise mutual information
    '''

    subj_verb_cnts = defaultdict(Counter)
    verb_cnts = Counter()
    pairs = 0

    for token in text:
        if token.dep_ in ("nsubj", "nsubjpass"):
            subj = token.text.lower()
            verb = token.head.lemma_.lower()
            subj_verb_cnts[subj][verb] += 1
            verb_cnts[verb] += 1
            pairs += 1

    targ_cnts = subj_verb_cnts[subject]
    targ_tot = sum(targ_cnts.values())

    for verb, cnt in targ_cnts.items():
        p_subj_verb = cnt / 